# module-modules-iter-isinstance-dispatch — worked example 3: Zero-initialize biases on all Conv2d layers via model.modules()

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `module-modules-iter-isinstance-dispatch`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import torch.nn as nn
import matplotlib.pyplot as plt

## Concept

After initializing a model, it is common to apply custom weight-initialization rules to every layer of a specific type. `model.modules()` combined with `isinstance` gives you a loop that reaches every `Conv2d` in the network no matter how deeply nested, and from there you can directly access and modify `.bias.data` in-place. This is the manual alternative to `model.apply()` when you need to branch on type.

## Worked solution

**Step 1 — walk with `.modules()`.** As always, this is recursive and yields every submodule.

**Step 2 — dispatch to Conv2d only.** Use `isinstance(m, nn.Conv2d)` as the guard. Only Conv2d layers have a `.bias` we want to zero.

**Step 3 — check that bias exists.** `nn.Conv2d` can be constructed with `bias=False`, in which case `m.bias` is `None`. Guard with `if m.bias is not None:` before touching `.data`.

**Step 4 — zero in-place with `.data.zero_()`.** Working on `.data` bypasses autograd, which is correct here — we're doing an explicit initialization, not a computation we want to differentiate through.

**Count the modifications** so the caller can verify the operation touched the expected number of layers.

In [ ]:
import torch
import torch.nn as nn

def zero_conv_biases(model: nn.Module) -> int:
    """Zero-initialize biases on every Conv2d in model. Returns count of layers modified."""
    count = 0
    for m in model.modules():
        if isinstance(m, nn.Conv2d):
            if m.bias is not None:
                m.bias.data.zero_()
                count += 1
    return count

# Build a small conv network with both biased and unbiased convs.
torch.manual_seed(7)
net = nn.Sequential(
    nn.Conv2d(3, 8, 3, padding=1),           # has bias
    nn.ReLU(),
    nn.Sequential(
        nn.Conv2d(8, 16, 3, padding=1),      # has bias
        nn.Conv2d(16, 16, 1, bias=False),    # NO bias
    ),
    nn.Conv2d(16, 1, 1),                     # has bias
)

# Confirm biases are non-zero before zeroing (with high probability).
print("Before:", [m.bias.data.norm().item() for m in net.modules()
                  if isinstance(m, nn.Conv2d) and m.bias is not None])

modified = zero_conv_biases(net)

# Confirm biases are now zero.
print("After: ", [m.bias.data.norm().item() for m in net.modules()
                  if isinstance(m, nn.Conv2d) and m.bias is not None])
print(f"Modified {modified} Conv2d layers with bias")  # 3